# Using boti_sweet_etl datacubes

Demonstrates `boti_sweet_etl.Datasources.datacube()` / `.data_helper()` — building a `boti_data.datacube.BaseDataCube` from a named SQL connection profile in `datasources.yaml`.

Run from `sandbox/notebooks/` with the workspace's own kernel (`uv run python -m ipykernel install --user --name boti-sweet`, then select "boti-sweet" as the kernel) — needs the `etl` extra synced (`uv sync --extra etl` or `--all-extras`).

In [1]:
import os
import sys
from pathlib import Path

SANDBOX_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(SANDBOX_DIR))

from deployment_settings import build_datasources  # noqa: E402 - needs sys.path.insert above first

os.environ.setdefault("BOTI_SWEET_CONFIG_DIR", str(SANDBOX_DIR / "config"))

'/Users/lvalverdeb/TeamDev/boti-sweet/sandbox/config'

## Self-contained example

Uses a temporary sqlite database (stdlib driver, no server, no real credentials) so this notebook always runs, independent of any real deployment config.

In [2]:
import sqlite3
import tempfile

demo_db = Path(tempfile.mkstemp(suffix=".db")[1])
conn = sqlite3.connect(demo_db)
conn.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, status TEXT, amount REAL)")
conn.executemany(
    "INSERT INTO orders (status, amount) VALUES (?, ?)",
    [("active", 100.0), ("active", 250.5), ("cancelled", 40.0)],
)
conn.commit()
conn.close()

demo_yaml = Path(tempfile.mkstemp(suffix=".yaml")[1])
demo_yaml.write_text(
    "sql:\n"
    "  connections:\n"
    "    demo:\n"
    f"      connection_url: \"sqlite:///{demo_db}\"\n"
    # sqlite can't enforce native read-only mode, so query_only=False here
    # (SqlDatabaseResource's own default, query_only=True, would fail fast)
    "      query_only: false\n"
)

demo_datasources = build_datasources(datasources_file=demo_yaml)

In [3]:
cube = demo_datasources.datacube("demo", table="orders")
df = cube.load(return_type="pandas")
df

,id,status,amount
0,1,active,100.0
1,2,active,250.5
2,3,cancelled,40.0


Configured-mode filter kwargs become `WHERE` conditions:

In [4]:
cube.load(return_type="pandas", status="active")

,id,status,amount
0,1,active,100.0
1,2,active,250.5


## Subclassing `BaseDataCube`

`BaseDataCube.from_helper()` builds the plain base class. Subclass it to add a `fix_data()`/`afix_data()` transform hook, applied automatically after `load()`/`aload()` — see `BaseDataCube`'s own docstring in `boti_data.datacube.base`.

In [5]:
from boti_sweet_etl import BaseDataCube


class ActiveOrdersCube(BaseDataCube):
    def fix_data(self, **kwargs):
        self.df = self.df[self.df["status"] == "active"]


active_cube = ActiveOrdersCube.from_helper(
    demo_datasources.data_helper("demo", table="orders")
)
active_cube.load(return_type="pandas")

,id,status,amount
0,1,active,100.0
1,2,active,250.5


In [6]:
cube.close()
active_cube.close()
demo_db.unlink()
demo_yaml.unlink()

## Using this deployment's real credentials

Copy `sandbox/config/datasources.yaml.example` to `sandbox/config/datasources.yaml` and fill in real values (gitignored — never commit it) to try this against a real connection. Building a datacube eagerly creates a SQL engine, so it fails fast if the dialect's DBAPI driver (e.g. `pymysql`, `psycopg2`) isn't installed — that's a deployment concern, not a `boti-sweet-etl` dependency.

In [7]:
real_datasources_file = SANDBOX_DIR / "config" / "datasources.yaml"
real_datasources = build_datasources(datasources_file=real_datasources_file)

try:
    replica_config = real_datasources.sql("replica")
except KeyError:
    print("replica not configured — see sandbox/config/datasources.yaml.example")
else:
    print(f"replica: query_only={replica_config.query_only} pool_size={replica_config.pool_size}")
    try:
        replica_cube = real_datasources.datacube("replica")
    except Exception as exc:  # noqa: BLE001 - demo: report and move on
        print(f"could not build replica datacube: {exc}")
    else:
        print("replica datacube ready — call .load(table=..., ...) to fetch data")
        replica_cube.close()

replica not configured — see sandbox/config/datasources.yaml.example
